In [1]:
# Parameters
UF = "AC"


## Análise de sazonalidade (Kruskal-Wallis / Markham Seasonality Index)

Este notebook realiza a análise de sazonalidade das concentrações médias mensais de
poluentes por estação, a partir dos arquivos hospedados em `MQAr_averages/mensal/{POLUENTE}/*.csv`.
Considera apenas anos com os 12 meses válidos e calcula, por estação:

- **Amplitude** — diferença entre a média mensal mais alta e a mais baixa;
- **Força relativa da sazonalidade** — quanto da variância total é explicada pelo padrão sazonal;
- **Índice de Markham (MSI)** — quão concentrada a sazonalidade é em poucos meses;
- Teste de Kruskal-Wallis entre meses (significância da diferença sazonal).

Para cada poluente, gera:

- `{poluente}_seasonality.csv` — tabela com as métricas de sazonalidade por estação;
- `{poluente}_stations.geojson` — pontos das estações com essas métricas, usados no mapa interativo;
- `manifest_seasonality.json` — resumo da execução (arquivos gerados e nº de pontos por poluente).

Os arquivos são salvos localmente em `_static/preprocessed_seasonality/`.

> **Conexão com o relatório:** este notebook é um pré-requisito da **Seção 4.3**
> (`secao_4/secao_4.3.ipynb`) — os GeoJSONs gerados aqui alimentam o mapa interativo da
> **Figura 35** (sazonalidade de CO, NO₂, SO₂, MP₂,₅, MP₁₀ e O₃). Execute este notebook
> primeiro, antes de abrir `secao_4.3.ipynb`.

In [2]:
# -*- coding: utf-8 -*-

import re
import json
import warnings
import calendar
import os
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import geopandas as gpd
from shapely.geometry import Point
from scipy.stats import kruskal

#Configurações de entradas e saídas
# Todos os dados de entrada estão hospedados remotamente; OUTPUT_DIR é local (onde os
# CSVs/GeoJSONs consolidados são salvos) e já compatível com o basePath usado em secao_4.3.ipynb.
BASE_FOLDER = "https://arquivos.lcqar.ufsc.br/data/databases/stations/MQAr_averages/mensal/"
STATIONS_FILE = "https://arquivos.lcqar.ufsc.br/data/databases/stations/Monitoramento_QAr_BR.csv"
OUTPUT_DIR = Path("../_static/preprocessed_seasonality")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#Lista de poluentes para realizar a análise
POLLUTANTS = ["CO", "NO2", "SO2", "MP10", "MP25", "O3"]

#Colunas usadas na tabela de metadados das estações
ST_COL_ID = "ID_MMA_COMPLETO"
ST_COL_LAT = "LATITUDE"
ST_COL_LON = "LONGITUDE"
ST_COL_NAME = "ID_OEMA"

FILE_EXT = ".csv"
CRS_OUT = "EPSG:4326"

#Função de análise de sazonalidade
def seasonality_analysis(df, col="VALOR", period=None, alpha=0.05):
    """
    Realiza análise de sazonalidade considerando apenas anos completos.

    Entrada
    -----
    df : DataFrame com colunas 'ano', 'mes' e `col`.
    col : nome da coluna com valores (default 'VALOR').
    period : tuple (ano_ini, ano_fim) ou None -> período a considerar.
    alpha : nível de significância para Kruskal-Wallis.

    Retorna
    -----
    dicionário com métricas da análise de sazonalidade.
    """
    if not {"ANO", "MES", col}.issubset(df.columns):
        raise ValueError("O DataFrame deve conter as colunas 'ano', 'mes' e a coluna de valores.")

    if period is None:
        start_year = int(df["ANO"].min())
        end_year = int(df["ANO"].max())
    else:
        start_year, end_year = map(int, period)

    df_period = df.loc[(df["ANO"] >= start_year) & (df["ANO"] <= end_year)].copy()

    #Identificar anos completos (12 meses válidos)
    valid_years = []
    invalid_years = []
    for year, group in df_period.groupby("ANO"):
        if group[col].notna().sum() == 12:
            valid_years.append(year)
        else:
            invalid_years.append(year)

    n_valid_years = len(valid_years)
    invalid_years_str = ", ".join(map(str, invalid_years)) if invalid_years else ""

    if n_valid_years == 0:
        return {
            "start_year": start_year,
            "end_year": end_year,
            "n_valid_years": 0,
            "invalid_years": invalid_years_str,
            "significance": False,
            "p_value": np.nan,
            "amplitude": np.nan,
            "relative_strength": np.nan,
            "msi": np.nan,
            "min_month": "",
            "max_month": "",
            "seasonal_index": {}
        }

    df_valid = df_period[df_period["ANO"].isin(valid_years)].copy()

    monthly_means = df_valid.groupby("MES")[col].mean()
    overall_mean = monthly_means.mean()
    seasonal_index = (monthly_means / overall_mean).to_dict()

    amplitude = float(monthly_means.max() - monthly_means.min())

    var_total = df_valid[col].var()
    var_seasonal = monthly_means.var()
    relative_strength = float(var_seasonal / var_total) if var_total > 0 else np.nan

    min_month_num = int(monthly_means.idxmin())
    max_month_num = int(monthly_means.idxmax())
    min_month = calendar.month_name[min_month_num]
    max_month = calendar.month_name[max_month_num]

    groups = [group[col].dropna().values for _, group in df_valid.groupby("MES")]
    if all(len(g) > 0 for g in groups):
        try:
            stat, p_value = kruskal(*groups)
            significant = bool(p_value < alpha)
        except Exception:
            p_value = np.nan
            significant = False
    else:
        p_value = np.nan
        significant = False

    values = monthly_means.values
    if np.nansum(values) > 0:
        rel_freq = values / np.nansum(values)
        msi = np.sum(np.abs(rel_freq - 1/12)) / (2 * (1 - 1/12))
    else:
        msi = np.nan

    return {
        "start_year": start_year,
        "end_year": end_year,
        "n_valid_years": n_valid_years,
        "invalid_years": invalid_years_str,
        "significance": significant,
        "p_value": float(p_value) if not pd.isna(p_value) else np.nan,
        "amplitude": amplitude,
        "relative_strength": relative_strength,
        "msi": msi,
        "min_month": min_month,
        "max_month": max_month,
        "seasonal_index": seasonal_index
    }

# Funções auxiliares para navegar diretórios remotos (o servidor expõe listagem via
# autoindex, então extraímos nomes de pastas/arquivos com regex a partir do HTML,
# já que HTTP não suporta os.listdir()/Path.iterdir() como um caminho local).
def list_remote_entries(url, pattern):
    resp = requests.get(url)
    resp.raise_for_status()
    return sorted(set(re.findall(pattern, resp.text)))

def list_remote_dirs(url):
    return list_remote_entries(url, r'href="([^"/]+)/"')

def list_remote_files(url, ext=".csv"):
    return list_remote_entries(url, r'href="([^"]+' + re.escape(ext) + r')"')

# Funções complementares
def find_pollutant_folder(base_folder, pol: str):
    try:
        dirs = list_remote_dirs(base_folder)
    except Exception as e:
        print(f"Falha ao listar {base_folder}: {e}")
        return None
    for d in dirs:
        if pol.lower() == d.lower() or pol.lower() in d.lower():
            return base_folder + d + "/"
    return None

def read_stations_meta(stations_file):
    try:
        df = pd.read_csv(stations_file)
    except Exception as e:
        warnings.warn(f"Não foi possível ler {stations_file}: {e}")
        return pd.DataFrame(columns=[ST_COL_ID, ST_COL_NAME, ST_COL_LAT, ST_COL_LON])
    for c in [ST_COL_ID, ST_COL_NAME, ST_COL_LAT, ST_COL_LON]:
        if c not in df.columns:
            df[c] = pd.NA
    df = df.drop_duplicates(subset=[ST_COL_ID])
    return df[[ST_COL_ID, ST_COL_NAME, ST_COL_LAT, ST_COL_LON]]

def merge_with_station_coords(season_df: pd.DataFrame, stations_meta: pd.DataFrame):
    if season_df.empty:
        return season_df
    if stations_meta.empty:
        season_df[ST_COL_LAT] = pd.NA
        season_df[ST_COL_LON] = pd.NA
        return season_df

    merged = season_df.merge(stations_meta, how="left", left_on="station", right_on=ST_COL_ID)
    if ST_COL_LAT not in merged.columns:
        merged[ST_COL_LAT] = pd.NA
    if ST_COL_LON not in merged.columns:
        merged[ST_COL_LON] = pd.NA
    return merged

def save_results_and_geojson(pol: str, merged_df: pd.DataFrame, out_dir: Path):
    pol_safe = pol.lower().replace(".", "").replace(" ", "_").replace(",", "")
    csv_path = out_dir / f"{pol_safe}_seasonality.csv"
    geojson_path = out_dir / f"{pol_safe}_stations.geojson"

    try:
        merged_df.to_csv(csv_path, index=False, encoding="utf-8")
    except Exception as e:
        print(f"Falha ao salvar CSV {csv_path}: {e}")

    n_points = 0
    if ST_COL_LAT in merged_df.columns and ST_COL_LON in merged_df.columns:
        mask = merged_df[ST_COL_LAT].notna() & merged_df[ST_COL_LON].notna()
        pts = merged_df.loc[mask].copy()
        if not pts.empty:
            try:
                pts[ST_COL_LAT] = pts[ST_COL_LAT].astype(float)
                pts[ST_COL_LON] = pts[ST_COL_LON].astype(float)
                pts["geometry"] = [Point(xy) for xy in zip(pts[ST_COL_LON], pts[ST_COL_LAT])]
                gdf = gpd.GeoDataFrame(pts, geometry="geometry", crs=CRS_OUT)
                gdf.to_file(geojson_path, driver="GeoJSON")
                n_points = len(gdf)
            except Exception as e:
                print(f"Falha ao gerar GeoJSON para {pol}: {e}")
                geojson_path.write_text(json.dumps({"type":"FeatureCollection","features":[]}, ensure_ascii=False))
                n_points = 0
        else:
            geojson_path.write_text(json.dumps({"type":"FeatureCollection","features":[]}, ensure_ascii=False))
            n_points = 0
    else:
        geojson_path.write_text(json.dumps({"type":"FeatureCollection","features":[]}, ensure_ascii=False))
        n_points = 0

    return {"pol": pol.lower(), "csv": csv_path.name, "geojson": geojson_path.name, "n_points": int(n_points)}

# Runner principal
def seasonality_analysis_folder(folder_url, col="VALOR", period=None, file_ext=".csv"):
    results = []

    try:
        filenames = list_remote_files(folder_url, ext=file_ext)
    except Exception as e:
        print(f"Falha ao listar {folder_url}: {e}")
        return pd.DataFrame(results)

    for filename in filenames:
        station_name = filename.rsplit(".", 1)[0]
        file_url = folder_url + filename

        try:
            df = pd.read_csv(file_url)
        except Exception as e:
            print(f"Falha ao ler {filename}: {e}")
            continue

        try:
            res = seasonality_analysis(df, col=col, period=period)
            res["station"] = station_name
            results.append(res)
        except Exception as e:
            print(f"Erro ao processar {filename}: {e}")

    return pd.DataFrame(results)


def process_all_seasonality(base_folder=BASE_FOLDER, stations_file=STATIONS_FILE,
                            pollutants=POLLUTANTS, out_dir: Path = OUTPUT_DIR, file_ext=FILE_EXT,
                            col="VALOR", period=None):
    manifest = {"generated": []}
    print("Base folder:", base_folder)

    stations_meta = read_stations_meta(stations_file)

    for pol in pollutants:
        pol_folder = find_pollutant_folder(base_folder, pol)
        if pol_folder is None:
            print(f"Pasta do poluente '{pol}' não encontrada em {base_folder}; pulando.")
            manifest["generated"].append({"pol": pol.lower(), "csv": None, "geojson": None, "n_points": 0})
            continue

        csv_files = list_remote_files(pol_folder, ext=file_ext)
        if not csv_files:
            print(f"Pasta encontrada para '{pol}' ({pol_folder}), mas sem arquivos {file_ext}; pulando.")
            manifest["generated"].append({"pol": pol.lower(), "csv": None, "geojson": None, "n_points": 0})
            continue

        print(f"Processando sazonalidade do poluente '{pol}' em: {pol_folder} (arquivos: {len(csv_files)})")

        season_df = seasonality_analysis_folder(pol_folder, col=col, period=period, file_ext=file_ext)

        if season_df.empty:
            print(f"Nenhum resultado válido para {pol} após análise; pulando geração de ficheiros.")
            manifest["generated"].append({"pol": pol.lower(), "csv": None, "geojson": None, "n_points": 0})
            continue

        merged = merge_with_station_coords(season_df, stations_meta)

        summary = save_results_and_geojson(pol, merged, out_dir)
        manifest["generated"].append(summary)
        print(f"{pol}: CSV -> {summary['csv']}, GeoJSON -> {summary['geojson']} (pontos: {summary['n_points']})")

    manifest_path = out_dir / "manifest_seasonality.json"
    try:
        with open(manifest_path, "w", encoding="utf-8") as fh:
            json.dump(manifest, fh, ensure_ascii=False, indent=2)
        print("Manifest salvo em:", manifest_path)
    except Exception as e:
        print("Falha ao salvar manifest:", e)

    return manifest

# Execução
if __name__ == "__main__":
    manifest = process_all_seasonality()
    print("Concluído. Manifest:", manifest)


Base folder: https://arquivos.lcqar.ufsc.br/data/databases/stations/MQAr_averages/mensal/


Processando sazonalidade do poluente 'CO' em: https://arquivos.lcqar.ufsc.br/data/databases/stations/MQAr_averages/mensal/CO/ (arquivos: 194)


CO: CSV -> co_seasonality.csv, GeoJSON -> co_stations.geojson (pontos: 174)
Processando sazonalidade do poluente 'NO2' em: https://arquivos.lcqar.ufsc.br/data/databases/stations/MQAr_averages/mensal/NO2/ (arquivos: 254)


NO2: CSV -> no2_seasonality.csv, GeoJSON -> no2_stations.geojson (pontos: 224)
Processando sazonalidade do poluente 'SO2' em: https://arquivos.lcqar.ufsc.br/data/databases/stations/MQAr_averages/mensal/SO2/ (arquivos: 202)


SO2: CSV -> so2_seasonality.csv, GeoJSON -> so2_stations.geojson (pontos: 183)
Processando sazonalidade do poluente 'MP10' em: https://arquivos.lcqar.ufsc.br/data/databases/stations/MQAr_averages/mensal/MP10/ (arquivos: 307)


MP10: CSV -> mp10_seasonality.csv, GeoJSON -> mp10_stations.geojson (pontos: 283)
Processando sazonalidade do poluente 'MP25' em: https://arquivos.lcqar.ufsc.br/data/databases/stations/MQAr_averages/mensal/MP25/ (arquivos: 220)


MP25: CSV -> mp25_seasonality.csv, GeoJSON -> mp25_stations.geojson (pontos: 182)
Processando sazonalidade do poluente 'O3' em: https://arquivos.lcqar.ufsc.br/data/databases/stations/MQAr_averages/mensal/O3/ (arquivos: 256)


O3: CSV -> o3_seasonality.csv, GeoJSON -> o3_stations.geojson (pontos: 231)
Manifest salvo em: ../_static/preprocessed_seasonality/manifest_seasonality.json
Concluído. Manifest: {'generated': [{'pol': 'co', 'csv': 'co_seasonality.csv', 'geojson': 'co_stations.geojson', 'n_points': 174}, {'pol': 'no2', 'csv': 'no2_seasonality.csv', 'geojson': 'no2_stations.geojson', 'n_points': 224}, {'pol': 'so2', 'csv': 'so2_seasonality.csv', 'geojson': 'so2_stations.geojson', 'n_points': 183}, {'pol': 'mp10', 'csv': 'mp10_seasonality.csv', 'geojson': 'mp10_stations.geojson', 'n_points': 283}, {'pol': 'mp25', 'csv': 'mp25_seasonality.csv', 'geojson': 'mp25_stations.geojson', 'n_points': 182}, {'pol': 'o3', 'csv': 'o3_seasonality.csv', 'geojson': 'o3_stations.geojson', 'n_points': 231}]}
